In [1]:
import pyspark

from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/16 10:06:30 WARN Utils: Your hostname, MarkABaltazar, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/16 10:06:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/16 10:06:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
df_green = spark.read.parquet('../data/pq/green/*/*')

26/06/16 10:07:43 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../data/pq/green/*/*.
java.io.FileNotFoundException: File ../data/pq/green/*/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.Resol

In [4]:
df_green.createOrReplaceTempView('green_data')

In [7]:
df_green.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- lpep_pickup_datetime: timestamp (nullable = true)
 |-- lpep_dropoff_datetime: timestamp (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- ehail_fee: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- trip_type: integer (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [19]:
df_green_rev = spark.sql("""
    SELECT 
        -- Revenue grouping 
        date_trunc('hour', lpep_pickup_datetime) AS hour,
        PULocationID AS revenue_zone,
    
        -- Revenue calculation 
        SUM(total_amount) AS revenue_monthly_total_amount,
        COUNT(1) AS member_records
    FROM
        green_data
    WHERE lpep_pickup_datetime >= '2020-01-01 00:00:00'
    GROUP BY
        hour, revenue_zone
    ORDER BY
        hour, revenue_zone
""")

In [18]:
df_green_rev.show()

[Stage 16:===========================================>              (6 + 2) / 8]

+-------------------+------------+----------------------------+--------------+
|               hour|revenue_zone|revenue_monthly_total_amount|member_records|
+-------------------+------------+----------------------------+--------------+
|2020-01-09 17:00:00|         225|                       82.46|             5|
|2020-01-02 21:00:00|          74|          506.42000000000013|            38|
|2020-01-27 13:00:00|          24|                      203.57|             7|
|2020-01-06 16:00:00|          43|          343.68000000000006|            21|
|2020-01-05 08:00:00|          55|                      132.52|             4|
|2020-01-20 15:00:00|          82|           380.2200000000001|            27|
|2020-01-28 22:00:00|         255|                      217.11|            13|
|2020-01-27 09:00:00|         177|                       102.8|             3|
|2020-01-26 20:00:00|          35|                       48.61|             2|
|2020-01-29 08:00:00|         223|          119.9299

In [20]:
df_green_rev.write.parquet('../data/report/revenue/green')

26/06/16 10:18:29 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [21]:
df_green_rev \
    .repartition(20) \
    .write.parquet('../data/report/revenue/green', mode="overwrite")

26/06/16 10:21:28 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/16 10:21:29 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [22]:
df_yellow = spark.read.parquet('../data/pq/yellow/*/*')

26/06/16 10:23:14 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../data/pq/yellow/*/*.
java.io.FileNotFoundException: File ../data/pq/yellow/*/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.Res

In [23]:
df_yellow.createOrReplaceTempView('yellow_data')

In [24]:
df_yellow.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [26]:
df_yellow_rev = spark.sql("""
    SELECT 
        -- Revenue grouping 
        date_trunc('hour', tpep_pickup_datetime) AS hour,
        PULocationID AS revenue_zone,
    
        -- Revenue calculation 
        SUM(total_amount) AS revenue_monthly_total_amount,
        COUNT(1) AS member_records
    FROM
        yellow_data
    WHERE tpep_pickup_datetime >= '2020-01-01 00:00:00'
    GROUP BY
        hour, revenue_zone
    ORDER BY
        hour, revenue_zone
""")

In [27]:
df_yellow_rev.show()

[Stage 40:===================================================>    (11 + 1) / 12]

+-------------------+------------+----------------------------+--------------+
|               hour|revenue_zone|revenue_monthly_total_amount|member_records|
+-------------------+------------+----------------------------+--------------+
|2020-01-01 00:00:00|           3|                        25.0|             1|
|2020-01-01 00:00:00|           4|          1004.3000000000003|            57|
|2020-01-01 00:00:00|           7|          455.17000000000013|            38|
|2020-01-01 00:00:00|          10|                       42.41|             2|
|2020-01-01 00:00:00|          12|          106.99999999999999|             6|
|2020-01-01 00:00:00|          13|          1214.8000000000002|            56|
|2020-01-01 00:00:00|          14|                         8.8|             1|
|2020-01-01 00:00:00|          15|                       34.09|             1|
|2020-01-01 00:00:00|          17|          220.20999999999998|             8|
|2020-01-01 00:00:00|          18|                  

In [28]:
df_yellow_rev \
    .repartition(20) \
    .write.parquet('../data/report/revenue/yellow')

26/06/16 10:27:27 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/06/16 10:27:28 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [31]:
df_trips = df_green_rev.join(df_yellow_rev, on=['hour','revenue_zone'], how='outer')

In [32]:
df_trips.show()

[Stage 56:===================================================>    (11 + 1) / 12]

+-------------------+------------+----------------------------+--------------+----------------------------+--------------+
|               hour|revenue_zone|revenue_monthly_total_amount|member_records|revenue_monthly_total_amount|member_records|
+-------------------+------------+----------------------------+--------------+----------------------------+--------------+
|2020-01-01 00:00:00|          34|                        NULL|          NULL|                        19.3|             1|
|2020-01-01 00:00:00|          61|                      526.71|            17|                      146.64|             3|
|2020-01-01 00:00:00|          65|          199.48999999999998|            10|                      409.35|            19|
|2020-01-01 00:00:00|          68|                        NULL|          NULL|           7825.070000000015|           396|
|2020-01-01 00:00:00|          70|          54.900000000000006|             3|                         9.3|             1|
|2020-01-01 00:0

In [33]:
df_green_rev_temp = df_green_rev \
    .withColumnRenamed('revenue_monthly_total_amount', 'green_total_amount') \
    .withColumnRenamed('member_records', 'green_member_records')

df_yellow_rev_temp = df_yellow_rev \
    .withColumnRenamed('revenue_monthly_total_amount', 'yellow_total_amount') \
    .withColumnRenamed('member_records', 'yellow_member_records')

In [34]:
df_trips = df_green_rev_temp.join(df_yellow_rev_temp, on=['hour','revenue_zone'], how='outer')

In [35]:
df_trips.show()

[Stage 61:===================================================>    (11 + 1) / 12]

+-------------------+------------+------------------+--------------------+-------------------+---------------------+
|               hour|revenue_zone|green_total_amount|green_member_records|yellow_total_amount|yellow_member_records|
+-------------------+------------+------------------+--------------------+-------------------+---------------------+
|2020-01-01 00:00:00|          34|              NULL|                NULL|               19.3|                    1|
|2020-01-01 00:00:00|          61|            526.71|                  17|             146.64|                    3|
|2020-01-01 00:00:00|          65|199.48999999999998|                  10|             409.35|                   19|
|2020-01-01 00:00:00|          68|              NULL|                NULL|  7825.070000000015|                  396|
|2020-01-01 00:00:00|          70|54.900000000000006|                   3|                9.3|                    1|
|2020-01-01 00:00:00|          74|317.09000000000015|           

In [36]:
df_trips.write.parquet('../data/report/revenue/total')

26/06/16 10:40:54 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [37]:
df_green_revenue = spark.read.parquet('../data/report/revenue/green')
df_yellow_revenue = spark.read.parquet('../data/report/revenue/yellow')

In [38]:
df_green_rev_temp = df_green_revenue \
    .withColumnRenamed('revenue_monthly_total_amount', 'green_total_amount') \
    .withColumnRenamed('member_records', 'green_member_records')

df_yellow_rev_temp = df_yellow_revenue \
    .withColumnRenamed('revenue_monthly_total_amount', 'yellow_total_amount') \
    .withColumnRenamed('member_records', 'yellow_member_records')

In [39]:
df_trips_revenue = df_green_rev_temp.join(df_yellow_rev_temp, on=['hour','revenue_zone'], how='outer')

In [40]:
df_trips_revenue.write.parquet('../data/report/revenue/total', mode="overwrite")

26/06/16 13:23:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [41]:
df_trips_revenue = spark.read.parquet('../data/report/revenue/total')

In [42]:
df_trips_revenue.show()

+-------------------+------------+------------------+--------------------+-------------------+---------------------+
|               hour|revenue_zone|green_total_amount|green_member_records|yellow_total_amount|yellow_member_records|
+-------------------+------------+------------------+--------------------+-------------------+---------------------+
|2020-01-01 00:00:00|           7| 769.7299999999996|                  45| 455.17000000000013|                   38|
|2020-01-01 00:00:00|          35|            129.96|                   5|               NULL|                 NULL|
|2020-01-01 00:00:00|          37|            175.67|                   6| 161.60999999999999|                    7|
|2020-01-01 00:00:00|          43|            107.52|                   6|  6539.510000000012|                  390|
|2020-01-01 00:00:00|          59|50.900000000000006|                   3|               NULL|                 NULL|
|2020-01-01 00:00:00|          80|            364.32|           

In [43]:
df_zones = spark.read.parquet('../zones/')

In [44]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [45]:
df_trips_revenue

DataFrame[hour: timestamp, revenue_zone: int, green_total_amount: double, green_member_records: bigint, yellow_total_amount: double, yellow_member_records: bigint]

In [46]:
df_zones

DataFrame[LocationID: string, Borough: string, Zone: string, service_zone: string]

In [47]:
df_result = df_trips_revenue.join(df_zones, df_trips_revenue.revenue_zone == df_zones.LocationID)

In [52]:
df_result.drop('LocationID', 'revenue_zone').write.parquet('../tmp/revenue-zones')

26/06/16 13:34:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                